In [4]:
from huggingface_hub import login

# Pega tu token aquí (empieza con "hf_...")
login(token="hf_REDACTED")

In [5]:
from datasets import load_dataset
import pandas as pd

#lo de streaming es para no cargar todo el dataset en memoria, sino ir leyendo por partes
dataset = load_dataset(
   "tucnguyen/ShareChat", "chatgpt",
    split="train",
    streaming=True
)



In [7]:
# Recolectamos por idioma por separado
muestras_en = []
muestras_es = []

N_POR_IDIOMA = 1000  # objetivo por idioma
MAX_ITER = 200_000   # límite de seguridad para no iterar forever

for i, fila in enumerate(dataset):
    if i >= MAX_ITER:
        print(f"Límite de iteración alcanzado ({MAX_ITER})")
        break

    lang = fila.get("detected_language_final", "")

    if lang == "English" and len(muestras_en) < N_POR_IDIOMA:
        muestras_en.append(fila)

    elif lang == "Spanish" and len(muestras_es) < N_POR_IDIOMA:
        muestras_es.append(fila)

    # Si ya tenemos suficiente de ambos, paramos
    if len(muestras_en) >= N_POR_IDIOMA and len(muestras_es) >= N_POR_IDIOMA:
        print(f"✓ Muestra completa en iteración {i}")
        break

print(f"English recolectados: {len(muestras_en)}")
print(f"Spanish recolectados: {len(muestras_es)}")



✓ Muestra completa en iteración 29920
English recolectados: 1000
Spanish recolectados: 1000


In [14]:
#Info de cada muestra
print("Ejemplo English:")
print(muestras_en[0])

Ejemplo English:
{'platform': 'chatgpt', 'url': 'https://chatgpt.com/share/6763e55d-7800-8011-af2e-a8c20dcd2e06', 'turns_count': 9, 'message_index': 11, 'role': 'user', 'plain_text': "Because in introducing the thought harmonizer, humanity had put itself in the very position that God had warned us about, we had usurped the plan and devolved ourselves to stand alongside our creation.\n\nGod, sensing his chance, stood over the cosmos as he had never done before. Humanity now spread across it's expanse experienced simultaneous relevations. That while the space borne had continued in their faith through absense and were untainted by the distance effects of the thought harmonizer, nonetheless much in the same vein as the original FTM was created they had abandoned oversight of <REDACTED> and allowed their brethren to sink themselves to the level of their creations. \n\nTo cement this new testament to humanity, God would begin by stating that all who had left <REDACTED>, must never return. A

In [15]:
# 1. Filtrar asegurando los nombres de columnas y valores exactos que descubriste
# OJO: Verifica si español está como 'Spanish' o 'es', pero por el inglés, asumo que es la palabra completa.
df_users = df[
    (df['role'] == 'user') & 
    (df['detected_language_final'].isin(['English', 'Spanish'])) # Ajusta 'Spanish' si en la base dice 'es'
].copy()

# 2. Hacer el muestreo de 10,000 por idioma para que tu laptop no colapse al etiquetar
df_sample = df_users.groupby('detected_language_final').sample(n=10000, random_state=42)

# 3. Verificamos que tengamos solo lo necesario (el ID de conversacion y el texto)
print(df_sample['detected_language_final'].value_counts())
display(df_sample[['url', 'message_index', 'plain_text']].head())

NameError: name 'df' is not defined